# CLR Evaluation Notebook
Unified evaluation notebook for all model types (pi0, pi05, xvla).
Set `MODEL_TYPE` below and the notebook will load the correct policy class and experiment config.

In [ ]:
#!pip uninstall -y transformers
#!pip install git+https://github.com/huggingface/transformers.git@fix/lerobot_openpi

In [ ]:
# @title Parameters
MODEL_TYPE = "pi0"  # @param ["pi0", "pi05", "xvla", "wall_x"]
EXPERIMENT_CONFIG = ""  # @param {"type":"string"}
# If EXPERIMENT_CONFIG is empty, defaults to experiment-{MODEL_TYPE}.cfg

In [ ]:
from datetime import datetime
import importlib
import random
import numpy as np
import os
import torch
import json
from PIL import Image
from src.env.env_clr import RILAB_OMY_ENV
from torchvision import transforms

from lerobot.processor import PolicyAction, PolicyProcessorPipeline
from lerobot.processor.converters import (
    batch_to_transition,
    policy_action_to_transition,
    transition_to_batch,
    transition_to_policy_action,
)
from lerobot.utils.constants import POLICY_POSTPROCESSOR_DEFAULT_NAME, POLICY_PREPROCESSOR_DEFAULT_NAME

# Register model-specific processor steps so they are available when loading from JSON
import lerobot.policies.wall_x.processor_wall_x  # noqa: F401 — registers wall_x_task_processor

import glfw

In [ ]:
POLICY_REGISTRY = {
    "pi0": ("lerobot.policies.pi0.modeling_pi0", "PI0Policy"),
    "pi05": ("lerobot.policies.pi05.modeling_pi05", "PI05Policy"),
    "xvla": ("lerobot.policies.xvla.modeling_xvla", "XVLAPolicy"),
    "wall_x": ("lerobot.policies.wall_x.modeling_wall_x", "WallXPolicy"),
}

if MODEL_TYPE not in POLICY_REGISTRY:
    raise ValueError(f"Unknown MODEL_TYPE '{MODEL_TYPE}'. Choose from: {list(POLICY_REGISTRY.keys())}")

module_path, class_name = POLICY_REGISTRY[MODEL_TYPE]
PolicyClass = getattr(importlib.import_module(module_path), class_name)
print(f"Using policy: {class_name} from {module_path}")

In [ ]:
import configparser
config = configparser.ConfigParser()
config_file = EXPERIMENT_CONFIG if EXPERIMENT_CONFIG else f"experiment-{MODEL_TYPE}.cfg"
config.read(config_file)
print(f"Loaded config: {config_file}")
exp = config["experiment"]

DATASET_ROOT = exp["DATASET_ROOT"]
DATASET_REPO = exp["DATASET_REPO"]
POLICY_REPO = exp["POLICY_REPO"]
OUTPUT_DIR = exp["OUTPUT_DIR"]
JOB_NAME = exp["JOB_NAME"] + datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
MAX_TRAIN_STEPS = int(exp["MAX_TRAIN_STEPS"])
CHUNK_SIZE = int(exp["CHUNK_SIZE"])
ACTION_STEPS = int(exp["ACTION_STEPS"])
BATCH_SIZE = int(exp["BATCH_SIZE"])

In [ ]:
print (f"DATASET_ROOT: {DATASET_ROOT}")
print (f"DATASET_REPO: {DATASET_REPO}")
print (f"POLICY_REPO: {POLICY_REPO}")
print (f"OUTPUT_DIR: {OUTPUT_DIR}")
print (f"JOB_NAME: {JOB_NAME}")
print (f"MAX_TRAIN_STEPS: {MAX_TRAIN_STEPS}")
print (f"CHUNK_SIZE: {CHUNK_SIZE}")
print (f"ACTION_STEPS: {ACTION_STEPS}")
print (f"BATCH_SIZE: {BATCH_SIZE}")

In [ ]:
'''
Load environment configuration and initialize environments
'''
# Evaluation Configuration
TEST_EPISODES = 10 #@param {"type":"integer"}
MAX_EPISODE_STEPS = 60_000 #@param {"type":"string"}
TASK=exp["TASK"]

## Load Model

In [ ]:
'''
Meta data is for loading dataset statistics and feature information
'''
device = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'

policy = PolicyClass.from_pretrained(POLICY_REPO)
_ = policy.to(device)

**Note**: If you want to change number of actions in chunk to be executed, please change:  

```policy.config.n_action_steps = YOUR_DESIRED_NUMBER```

In [ ]:
# Set the number of action steps to run in the environment for one invocation of the policy
policy.config.n_action_steps = int(exp["ACTION_STEPS"])

In [ ]:
# Check normalization stats dimension
preprocessor = PolicyProcessorPipeline.from_pretrained(
                pretrained_model_name_or_path=POLICY_REPO,
                config_filename= f"{POLICY_PREPROCESSOR_DEFAULT_NAME}.json",
                overrides={"device_processor": {"device": device}},
                to_transition=batch_to_transition,
                to_output=transition_to_batch,
            )

for step in preprocessor.steps:
    if hasattr(step, "stats"):
        if "observation.state" in step.stats:
            print(f"Stats dimension for observation.state: {step.stats['observation.state']['mean'].shape}")

postprocessor = PolicyProcessorPipeline.from_pretrained(
                pretrained_model_name_or_path=POLICY_REPO,
                config_filename= f"{POLICY_POSTPROCESSOR_DEFAULT_NAME}.json",
                overrides={"device_processor": {"device": device}},
                to_transition=policy_action_to_transition,
                to_output=transition_to_policy_action,
            )

In [ ]:
batch = {
    'observation.state': np.zeros((1, 7), dtype=np.float32),
    'observation.image': np.zeros((1, 3, 448, 448), dtype=np.float32),
    'observation.wrist_image': np.zeros((1, 3, 448, 448), dtype=np.float32),
    'observation.left_scene_image': np.zeros((1, 3, 448, 448), dtype=np.float32),
    'observation.right_scene_image': np.zeros((1, 3, 448, 448), dtype=np.float32),
    'task': [TASK]
}
if MODEL_TYPE == "wall_x":
    batch['observation.eef_pose'] = np.zeros((1, 7), dtype=np.float32)
batch = preprocessor(batch)  # to initialize the processors
batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

_ = policy.select_action(batch)  # to initialize the model

## Load Environment

In [ ]:
config_file_path = './configs/train_clr.json'
with open(config_file_path) as f:
    env_conf = json.load(f)
omy_env = RILAB_OMY_ENV(cfg=env_conf, seed=0, 
                        action_type='joint', 
                        obs_type='joint_pos',
                        vis_mode = 'teleop')

In [ ]:
def get_default_transform():
    """
    Returns a torchvision transform that:
     Converts to a FloatTensor and scales pixel values [0,255] -> [0.0,1.0]
    """
    return transforms.Compose([
        transforms.ToTensor(),  # PIL [0–255] -> FloatTensor [0.0–1.0], shape C×H×W
    ])
IMG_TRANSFORM = get_default_transform()

## Evaluate

In [ ]:
'''
Run one evaluation episode
'''
def run_one_episode():
    omy_env.reset(leader_pose=True)
    policy.reset()
    observation = omy_env.get_observation()
    omy_env.env.tick = 0
    success = False
    while omy_env.env.is_viewer_alive() and omy_env.env.tick < MAX_EPISODE_STEPS:
        omy_env.step_env()
        if omy_env.env.loop_every(HZ = 20):
            success = omy_env.check_success()
            if success: break
            if omy_env.env.is_key_pressed_once(glfw.KEY_Z):
                break  # for debugging: press 'z' to end the episode
            
            agent_image, wrist_image, left_scene_image, right_scene_image = omy_env.grab_image()
            
            frame = {
                "observation.state": observation[:7].astype(np.float32),
                'task': [TASK]
            }
            
            # wall_x additionally requires end-effector pose
            if MODEL_TYPE == "wall_x":
                eef_pose = omy_env.get_ee_pose()  # [x, y, z, roll, pitch, yaw]
                gripper_state = np.array([1.0 if observation[6] > 0.5 else 0.0], dtype=np.float32)
                frame["observation.eef_pose"] = np.concatenate([eef_pose, gripper_state]).astype(np.float32)
            
            agent_image = Image.fromarray(agent_image)
            wrist_image = Image.fromarray(wrist_image)
            left_scene_image = Image.fromarray(left_scene_image)
            right_scene_image = Image.fromarray(right_scene_image)
            
            agent_image = agent_image.resize((448, 448))
            wrist_image = wrist_image.resize((448, 448))
            left_scene_image = left_scene_image.resize((448, 448))
            right_scene_image = right_scene_image.resize((448, 448))
            
            agent_image = IMG_TRANSFORM(agent_image)
            wrist_image = IMG_TRANSFORM(wrist_image)
            left_scene_image = IMG_TRANSFORM(left_scene_image)
            right_scene_image = IMG_TRANSFORM(right_scene_image)
            
            frame["observation.image"] = agent_image
            frame["observation.wrist_image"] = wrist_image
            frame["observation.left_scene_image"] = left_scene_image
            frame["observation.right_scene_image"] = right_scene_image
            
            # numpy to torch
            frame = {k: torch.tensor(v, dtype=torch.float32).unsqueeze(0) if isinstance(v, np.ndarray) else v for k, v in frame.items()}
            # pre-process the frame
            frame = preprocessor(frame)
            # move to device
            frame = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in frame.items()}
            # select action
            action = policy.select_action(frame)
            # post-process the action
            action = postprocessor(action)
            action = action.squeeze(0).cpu().float().numpy()
            observation = omy_env.step(action, gripper_mode='binary')
            omy_env.render()
    return success

In [ ]:
'''
Run evaluation over multiple episodes
'''
results = []
for episode in range(TEST_EPISODES):
    success = run_one_episode()
    results.append(success)
    print(f"Episode {episode+1}/{TEST_EPISODES} - Success: {success}")
omy_env.env.close_viewer()
# log average success rate
avg_success = sum(results) / len(results)
print(f"Average Success Rate over {TEST_EPISODES} episodes: {avg_success*100:.2f}%")